# 03 — DSOI Opportunity Model
**Project:** Dark Store Placement Intelligence
**Author:** Ishaan Mankotia

Computes Dark Store Opportunity Index and identifies top 10 expansion zones.

In [ ]:
import sys
sys.path.insert(0, '../src')
from data_loader import fetch_dark_stores, generate_demand_grid
from opportunity_model import (compute_nearest_store_distance, flag_covered_zones,
                               compute_dsoi, get_top_expansion_zones, estimate_order_demand)
import pandas as pd
import numpy as np

## 3.1 Run Full Scoring Pipeline

In [ ]:
stores = fetch_dark_stores()
zones  = generate_demand_grid()
zones  = compute_nearest_store_distance(zones, stores)
zones  = flag_covered_zones(zones, coverage_radius_km=1.5)
zones  = compute_dsoi(zones)
zones[zones['dsoi_score']>0]['dsoi_score'].describe().round(2)

## 3.2 Top 10 Expansion Zones

In [ ]:
top10  = get_top_expansion_zones(zones, n=10)
demand = estimate_order_demand(top10)

## 3.3 Sensitivity: Weight Scenarios

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

scenarios = {
    'Baseline (35/35/20/10)':        (0.35, 0.35, 0.20, 0.10),
    'Distance-heavy (25/45/20/10)':  (0.25, 0.45, 0.20, 0.10),
    'Income-heavy (30/30/35/05)':    (0.30, 0.30, 0.35, 0.05),
    'Density-heavy (45/35/15/05)':   (0.45, 0.35, 0.15, 0.05),
}

scaler = MinMaxScaler()
unserved = zones[~zones['store_covered']].copy()
unserved['dn'] = scaler.fit_transform(unserved[['population_density']])
unserved['dd'] = scaler.fit_transform(unserved[['nearest_store_dist_km']])
unserved['di'] = scaler.fit_transform(unserved[['income_proxy']])
unserved['dr'] = scaler.fit_transform(unserved[['delivery_demand']])

results = {}
for name, (w1,w2,w3,w4) in scenarios.items():
    scores = w1*unserved['dn'] + w2*unserved['dd'] + w3*unserved['di'] + w4*unserved['dr']
    results[name] = unserved.loc[scores.nlargest(10).index, 'zone_id'].tolist()

all_zones = [z for v in results.values() for z in v]
stable = [z for z,c in Counter(all_zones).items() if c == 4]
print(f'Zones stable across ALL scenarios: {stable}')
pd.DataFrame(results)